In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score



In [ ]:
# Load the cleaned dataset generated from the data cleaning step
df = pd.read_csv("music_cleaned.csv")

# Display basic structure of the dataset
print("Dataset shape:", df.shape)
df.head()

In [ ]:
# Display information about data types and non-null counts
print(df.info())

# Display summary statistics for numerical variables
print(df.describe())

In [ ]:
unique_keys = df['music_genre'].unique()
print(unique_keys)
X = df.values
print(df.dtypes)
print("Shape of the data:", X.shape)

In [ ]:
# Visualize the distribution of the target variable (music genre)
plt.figure(figsize=(8,5))
sns.countplot(x="music_genre", data=df)

plt.title("Distribution of Music Genres")
plt.xlabel("Music Genre")
plt.ylabel("Count")
plt.xticks(rotation=45)

plt.show()

In [ ]:
# Select numerical columns
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns

# Plot histograms for all numerical features
df[numeric_cols].hist(figsize=(12,10))

plt.suptitle("Distribution of Numerical Features")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation analysis among numerical features

numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns

# Remove instance_id and key because it is only an identifier and not meaningful for correlation analysis
numeric_cols = [col for col in numeric_cols if col not in ["instance_id", "key"]]

corr = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)

plt.title("Correlation Matrix of Numerical Features")
plt.tight_layout()
plt.show()

In [ ]:
# Find highly correlated feature pairs

corr_pairs = corr.abs().unstack().sort_values(ascending=False)

# Remove self-correlations
corr_pairs = corr_pairs[corr_pairs < 1]

# Keep only one direction of each pair
corr_pairs = corr_pairs.drop_duplicates()

print("Top correlated feature pairs:")
print(corr_pairs.head(10))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18,10))

# Row 1
sns.boxplot(x="music_genre", y="energy", data=df, ax=axes[0,0], palette="Set2")
axes[0,0].set_title("Energy")

sns.boxplot(x="music_genre", y="danceability", data=df, ax=axes[0,1], palette="Set3")
axes[0,1].set_title("Danceability")

sns.boxplot(x="music_genre", y="loudness", data=df, ax=axes[0,2], palette="coolwarm")
axes[0,2].set_title("Loudness")

sns.boxplot(x="music_genre", y="acousticness", data=df, ax=axes[0,3], palette="Set1")
axes[0,3].set_title("Acousticness")

# Row 2（可以再加几个）
sns.boxplot(x="music_genre", y="valence", data=df, ax=axes[1,0], palette="Set2")
axes[1,0].set_title("Valence")

sns.boxplot(x="music_genre", y="speechiness", data=df, ax=axes[1,1], palette="Set3")
axes[1,1].set_title("Speechiness")

sns.boxplot(x="music_genre", y="tempo", data=df, ax=axes[1,2], palette="coolwarm")
axes[1,2].set_title("Tempo")

fig.delaxes(axes[1,3])

# Rotate x labels
for row in axes:
    for ax in row:
        if ax:
            ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# No outliers
fig, axes = plt.subplots(1, 4, figsize=(18,5))

sns.boxplot(x="music_genre", y="energy", data=df, ax=axes[0], palette="Set2", showfliers=False)

sns.boxplot(x="music_genre", y="danceability", data=df, ax=axes[1], palette="Set3", showfliers=False)

sns.boxplot(x="music_genre", y="loudness", data=df, ax=axes[2], palette="coolwarm", showfliers=False)

sns.boxplot(x="music_genre", y="acousticness", data=df, ax=axes[3], palette="Set1", showfliers=False)

for ax in axes:
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance using only numerical features

# Select only numerical columns
X_num = df.select_dtypes(include=["int64", "float64"]).copy()

# Remove non-predictive identifier column if present
if "key" in X_num.columns:
    X_num = X_num.drop(columns=["key"])

# Remove non-predictive identifier column if present
if "instance_id" in X_num.columns:
    X_num = X_num.drop(columns=["instance_id"])

# Remove target if it was encoded as numeric by accident
if "music_genre" in X_num.columns:
    X_num = X_num.drop(columns=["music_genre"])

y = df["music_genre"]

# Fit Random Forest model
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_num, y)

# Store feature importance
importances = pd.Series(
    rf.feature_importances_,
    index=X_num.columns
).sort_values()

# Plot feature importance
plt.figure(figsize=(8,6))
importances.plot(kind="barh")

plt.title("Feature Importance Based on Numerical Features")
plt.xlabel("Importance")
plt.ylabel("Feature")

plt.tight_layout()
plt.show()

In [ ]:
# Analyze categorical variables (object-type columns)
for column in df.columns:
    if df[column].dtype == 'object':
        print(f"{column}: {df[column].nunique()} unique labels")

In [ ]:
# Count number of unique artists and tracks
print("Number of unique artists:", df['artist_name'].nunique())
print("Number of unique tracks:", df['track_name'].nunique())

In [ ]:
# Encode key for EDA
key_mapping = {
    'C': 0, 'C#': 1, 'D': 2, 'D#': 3, 'E': 4, 'F': 5, 
    'F#': 6, 'G': 7, 'G#': 8, 'A': 9, 'A#': 10, 'B': 11
}
df["key_encoded"] = df["key"].map(key_mapping)


In [ ]:
# Unsupervised Learning： dimensionality reduction, clustering

In [ ]:
# Numerical audio features for EDA and unsupervised learning 
audio_features = [
    "popularity", "acousticness", "danceability", "duration_ms",
    "energy", "instrumentalness", "key_encoded", "liveness",
    "loudness", "speechiness", "tempo", "valence"
]

X = df[audio_features].copy()
# safety check na values
if X.isnull().sum().sum() > 0:
    X = X.fillna(X.median())

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# PCA explained variance
pca_full = PCA()
pca_full.fit(X_scaled)

plt.figure(figsize=(8, 5))
plt.plot(
    range(1, len(pca_full.explained_variance_ratio_) + 1),
    np.cumsum(pca_full.explained_variance_ratio_),
    marker="o"
)
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("Cumulative Explained Variance by PCA")
plt.grid(True)
plt.show()

In [ ]:
# PCA : reduce features to 2D
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
pca_df["music_genre"] = df["music_genre"].values

plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="music_genre",
    alpha=0.5,
    s=20
)
plt.title("PCA Projection of Songs by Music Genre")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# tSNE

# Sampling 5000 observations
sample_df = df.sample(n=5000, random_state=42)
X_sample = sample_df[audio_features].copy()
X_sample = X_sample.fillna(X_sample.median())
X_sample_scaled = scaler.fit_transform(X_sample)

tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="pca",
    random_state=42
)

X_tsne = tsne.fit_transform(X_sample_scaled)

tsne_df = pd.DataFrame(X_tsne, columns=["TSNE1", "TSNE2"])
tsne_df["music_genre"] = sample_df["music_genre"].values

plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=tsne_df,
    x="TSNE1",
    y="TSNE2",
    hue="music_genre",
    alpha=0.6,
    s=20
)
plt.title("t-SNE Visualization of Music Audio Features")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# K-Means

# k = 2, 3, 4, ..., 10
k_values = range(2, 11)
inertias = []
silhouette_scores = []

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))

# Elbow plot
plt.figure(figsize=(8, 5))
plt.plot(k_values, inertias, marker="o")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method for K-means")
plt.grid(True)
plt.show()

# Silhouette score plot
plt.figure(figsize=(8, 5))
plt.plot(k_values, silhouette_scores, marker="o")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Scores for K-means")
plt.grid(True)
plt.show()

In [ ]:
# KMeans visualization on PCA space

# Select best k on silhouette score
best_k = k_values[np.argmax(silhouette_scores)]

# Fit final K-means
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

pca_df["cluster"] = cluster_labels

plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="cluster",
    palette="tab10",
    alpha=0.5,
    s=20
)
plt.title(f"K-means Clusters Visualized in PCA Space (k={best_k})")
plt.tight_layout()
plt.show()

In [ ]:
# Cluster and genre
cluster_genre_table = pd.crosstab(
    pca_df["cluster"],
    pca_df["music_genre"],
    normalize="index"
)

# Cluster vs Genre heatmap
plt.figure(figsize=(12, 6))
sns.heatmap(cluster_genre_table, cmap="Blues", annot=False)
plt.title("Genre Composition Within Each K-means Cluster")
plt.xlabel("Music Genre")
plt.ylabel("Cluster")
plt.tight_layout()
plt.show()

cluster_genre_table